In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Ekstraksi data dari file .mseed ke format JSON siap untuk MCU-Quake (3 komponen).
- Menggunakan STA/LTA untuk deteksi P-wave arrival dari komponen Z.
- Ekstrak 7 detik sinyal dan noise untuk setiap komponen (Z, N, E).
- Preprocessing: detrend, resample ke 100 Hz, normalisasi per komponen.
- Output: JSON dengan key 'Z','N','E','Z_noise','N_noise','E_noise'.
"""

import os
import sys
import json
import numpy as np
import pandas as pd
from pathlib import Path
from obspy import read, UTCDateTime
from obspy.signal.trigger import recursive_sta_lta
from tqdm import tqdm
import logging

# =============================================
# 1. KONFIGURASI
# =============================================

WAVEFORM_DIR = "/Volumes/Extreme SSD/unduhan_waveform_geofon"
OUTPUT_JSON = "/Volumes/Extreme SSD/unduhan_waveform_geofon/extracted_data_3comp.json"

# Parameter preprocessing
SAMPLE_RATE = 100.0
SIG_DURATION = 7.0
NOISE_DURATION = 7.0
NORM_WINDOW = 9.0

# Parameter STA/LTA
STA_WIN = 1.0
LTA_WIN = 10.0
TRIGGER_THRESHOLD = 3.5

# Jumlah file untuk testing (None untuk semua)
MAX_FILES = None  # misal 100 untuk testing

# =============================================
# 2. SETUP LOGGING
# =============================================

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(sys.stdout),
        logging.FileHandler("extract_waveforms_3comp.log")
    ]
)
logger = logging.getLogger(__name__)

# =============================================
# 3. FUNGSI PREPROCESSING
# =============================================

def pick_p_arrival(trace, search_window=15):
    """
    Deteksi P-wave arrival menggunakan STA/LTA pada trace Z.
    Kembalikan UTCDateTime dari pick pertama.
    """
    try:
        tr = trace.copy()
        sr = tr.stats.sampling_rate
        sta_n = int(STA_WIN * sr)
        lta_n = int(LTA_WIN * sr)
        
        if len(tr.data) < lta_n + sta_n:
            return tr.stats.starttime + 5.0
        
        cft = recursive_sta_lta(tr.data, sta_n, lta_n)
        trigger_indices = np.where(cft > TRIGGER_THRESHOLD)[0]
        
        if len(trigger_indices) > 0:
            pick_idx = trigger_indices[0]
            if pick_idx > int(2 * sr):
                return tr.stats.starttime + pick_idx / sr
        
        max_idx = np.argmax(np.abs(tr.data))
        if max_idx > 0:
            return tr.stats.starttime + max_idx / sr
    except Exception as e:
        logger.debug(f"Picking error: {e}")
        return trace.stats.starttime + 5.0
    
    return trace.stats.starttime + 5.0

def extract_component_windows(trace, p_arrival_time):
    """
    Ekstraksi 7 detik sinyal dan noise untuk satu trace komponen.
    Return (signal_list, noise_list) atau (None, None) jika gagal.
    """
    try:
        # Potong sinyal 7 detik setelah P
        sig_start = p_arrival_time
        sig_end = p_arrival_time + SIG_DURATION
        tr_signal = trace.copy().trim(sig_start, sig_end)
        
        # Potong noise 7 detik sebelum P
        noise_start = p_arrival_time - NOISE_DURATION
        noise_end = p_arrival_time
        tr_noise = trace.copy().trim(noise_start, noise_end)
        
        # Detrend
        tr_signal.detrend('simple')
        tr_noise.detrend('simple')
        
        # Resample ke 100 Hz
        if tr_signal.stats.sampling_rate != SAMPLE_RATE:
            tr_signal.resample(SAMPLE_RATE)
        if tr_noise.stats.sampling_rate != SAMPLE_RATE:
            tr_noise.resample(SAMPLE_RATE)
        
        # Normalisasi dengan max absolut 9 detik setelah P
        tr_norm = trace.copy().trim(p_arrival_time, p_arrival_time + NORM_WINDOW)
        if len(tr_norm.data) > 0:
            max_val = np.max(np.abs(tr_norm.data))
        else:
            max_val = np.max(np.abs(tr_signal.data))
        if max_val == 0:
            max_val = 1.0
        
        signal_data = tr_signal.data / max_val
        noise_data = tr_noise.data / max_val
        
        # Potong/padding ke 700 sampel
        target_len = int(SAMPLE_RATE * SIG_DURATION)
        def fix_length(data):
            if len(data) > target_len:
                return data[:target_len]
            elif len(data) < target_len:
                return np.pad(data, (0, target_len - len(data)), 'constant')
            return data
        
        return fix_length(signal_data).tolist(), fix_length(noise_data).tolist()
    except Exception as e:
        logger.debug(f"Extraction error for {trace.stats.channel}: {e}")
        return None, None

def process_file(file_path):
    """
    Proses satu file .mseed, return dict dengan semua komponen atau None.
    """
    try:
        st = read(str(file_path))
        if len(st) == 0:
            return None
        
        # Cari trace Z untuk P-pick
        trace_z = None
        for tr in st:
            if tr.stats.channel.endswith('Z'):
                trace_z = tr
                break
        if trace_z is None:
            # Jika tidak ada Z, ambil trace pertama dan catat warning
            trace_z = st[0]
            logger.debug(f"{file_path.name}: Tidak ada komponen Z, pakai {trace_z.stats.channel} untuk pick")
        
        # Deteksi P-wave dari trace Z (atau fallback)
        p_time = pick_p_arrival(trace_z)
        
        # Ekstrak untuk setiap komponen yang tersedia
        result = {
            'Z': None, 'N': None, 'E': None,
            'Z_noise': None, 'N_noise': None, 'E_noise': None,
            'network': None, 'station': None, 'p_arrival': str(p_time), 'file': file_path.name
        }
        
        # Ambil info network/station dari trace pertama
        if len(st) > 0:
            result['network'] = st[0].stats.network
            result['station'] = st[0].stats.station
        
        # Proses setiap trace
        for tr in st:
            comp = tr.stats.channel[-1]  # ambil huruf terakhir (Z, N, E)
            if comp not in ['Z', 'N', 'E']:
                # Jika channel tidak berakhiran Z/N/E, coba gunakan 2 huruf terakhir
                if len(tr.stats.channel) >= 2:
                    comp = tr.stats.channel[-2:]  # misal 'HZ' -> 'Z'
                    if comp.endswith('Z'): comp = 'Z'
                    elif comp.endswith('N'): comp = 'N'
                    elif comp.endswith('E'): comp = 'E'
                    else:
                        continue
                else:
                    continue
            
            signal, noise = extract_component_windows(tr, p_time)
            if signal is not None and noise is not None:
                result[f'{comp}'] = signal
                result[f'{comp}_noise'] = noise
        
        # Cek apakah minimal ada satu komponen yang berhasil
        if any(result[c] is not None for c in ['Z', 'N', 'E']):
            return result
        else:
            logger.warning(f"{file_path.name}: Tidak ada komponen yang berhasil diekstrak.")
            return None
    except Exception as e:
        logger.debug(f"Error processing {file_path.name}: {e}")
        return None

def main():
    logger.info("="*60)
    logger.info("🚀 EKSTRAKSI WAVEFORM 3 KOMPONEN KE JSON")
    logger.info("="*60)
    
    # Cari semua file .mseed
    wave_dir = Path(WAVEFORM_DIR)
    all_files = list(wave_dir.glob("*.mseed"))
    logger.info(f"📁 Ditemukan {len(all_files)} file .mseed")
    
    if MAX_FILES and len(all_files) > MAX_FILES:
        all_files = all_files[:MAX_FILES]
        logger.info(f"⚠️ Hanya memproses {MAX_FILES} file pertama.")
    
    # Load JSON yang sudah ada (resume)
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r') as f:
            existing_data = json.load(f)
        logger.info(f"📂 Load JSON existing: {len(existing_data)} entries")
    else:
        existing_data = {}
    
    # Proses file
    success = 0
    failed = 0
    skipped = 0
    
    for file_path in tqdm(all_files, desc="Memproses"):
        # Cek apakah file sudah ada di JSON (gunakan nama file sebagai key)
        if file_path.stem in existing_data:
            skipped += 1
            continue
        
        result = process_file(file_path)
        if result:
            # Gunakan nama file sebagai key
            key = file_path.stem
            existing_data[key] = {
                'type': 'se',
                'Z': result.get('Z'),
                'N': result.get('N'),
                'E': result.get('E'),
                'Z_noise': result.get('Z_noise'),
                'N_noise': result.get('N_noise'),
                'E_noise': result.get('E_noise'),
                'metadata': {
                    'network': result['network'],
                    'station': result['station'],
                    'p_arrival': result['p_arrival'],
                    'file': result['file']
                }
            }
            success += 1
        else:
            failed += 1
        
        # Simpan setiap 100 file
        if (success + failed) % 100 == 0:
            with open(OUTPUT_JSON, 'w') as f:
                json.dump(existing_data, f, indent=2)
    
    # Simpan final
    with open(OUTPUT_JSON, 'w') as f:
        json.dump(existing_data, f, indent=2)
    
    logger.info("="*60)
    logger.info(f"✨ SELESAI! Berhasil: {success}, Gagal: {failed}, Skipped: {skipped}")
    logger.info(f"📁 Total data di JSON: {len(existing_data)}")
    logger.info(f"📂 Output: {OUTPUT_JSON}")
    logger.info("="*60)

if __name__ == "__main__":
    main()

2026-06-21 09:40:57,439 - INFO - ============================================================
2026-06-21 09:40:57,440 - INFO - 🚀 EKSTRAKSI WAVEFORM 3 KOMPONEN KE JSON
2026-06-21 09:40:57,440 - INFO - ============================================================
2026-06-21 09:40:57,562 - INFO - 📁 Ditemukan 25880 file .mseed


Memproses:  42%|████▏     | 10974/25880 [13:47<21:54, 11.34it/s]  /opt/homebrew/Caskroom/miniforge/base/envs/tf-metal/lib/python3.11/site-packages/obspy/signal/detrend.py:31: RuntimeWarning: invalid value encountered in divide
  data -= x1 + np.arange(ndat) * (x2 - x1) / float(ndat - 1)
Memproses:  87%|████████▋ | 22586/25880 [58:05<05:48,  9.46it/s]  